# 04 · Camada ouro
Une alertas e respostas numa tabela com o status do funil de cada alerta
e as métricas de tempo, pronta para o Power BI.

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path

In [ ]:
# Caminhos do projeto (relativos à pasta notebooks/)
bronze_dir = Path("../dados/bronze")
silver_dir = Path("../dados/prata")
gold_dir = Path("../dados/ouro")
source_file = bronze_dir / "Controle de alertas - CMF-SEAMA.xlsx"

# Data de referência: última atualização da planilha (cabeçalho da aba DADOS)
header = pd.read_excel(source_file, sheet_name="DADOS", nrows=0).columns
reference_date = next(pd.Timestamp(c) for c in header if isinstance(c, datetime))
print("Dados atualizados em", reference_date.date())

In [ ]:
df_alerts = pd.read_parquet(silver_dir / "alertas.parquet")
df_responses = pd.read_parquet(silver_dir / "respostas.parquet")

# Desfecho de cada linha de resposta, na ordem de precedência
row_conditions = [
    df_responses["houve_infracao"].eq("sim"),
    df_responses["houve_infracao"].eq("não"),
    df_responses["enviado_fiscalizacao"].eq("sim"),
    df_responses["motivo"].eq("falso positivo"),
    df_responses["motivo"].isin(["alerta tardio", "iuf tardio"]),
    df_responses["motivo"].eq("terra indígena"),
]
row_labels = ["fiscalizado com infração", "fiscalizado sem infração",
              "aguardando fiscalização", "falso positivo",
              "tardio (a definir)", "outro órgão"]

df_responses["status"] = np.select([c.fillna(False).astype(bool) for c in row_conditions],
                                   row_labels, default="triado sem desfecho")

# Por alerta, vale o desfecho mais avançado (menor posição na lista)
priority = {label: rank for rank, label in enumerate(row_labels + ["triado sem desfecho"])}
df_responses["prioridade"] = df_responses["status"].map(priority)
df_status = (df_responses.sort_values("prioridade")
             .drop_duplicates("id_alerta")[["id_alerta", "status"]])

# Primeira fiscalização de cada alerta
first_inspection = df_responses.groupby("id_alerta", as_index=False)["data_fiscalizacao"].min()

In [ ]:
# Junta tudo na tabela de alertas, garantindo que nenhuma junção multiplique linhas
df_gold = (df_alerts
           .merge(df_status, on="id_alerta", how="left", validate="one_to_one")
           .merge(first_inspection, on="id_alerta", how="left", validate="one_to_one"))
df_gold["status"] = df_gold["status"].fillna("não triado")

# Alertas que têm resposta na planilha, mas só em linhas de quarentena
in_quarantine = df_gold["tem_resposta"] & df_gold["status"].eq("não triado")
df_gold.loc[in_quarantine, "status"] = "resposta em quarentena"

assert len(df_gold) == len(df_alerts), "A junção mudou o número de alertas"
print(df_gold["status"].value_counts().to_string())

In [ ]:
# Renomeia o status que descreve ausência de registro, não ausência de triagem
df_gold["status"] = df_gold["status"].replace({"não triado": "sem registro de resposta"})

# Dias entre a publicação do alerta e a primeira fiscalização
df_gold["dias_ate_fiscalizacao"] = (df_gold["data_fiscalizacao"] - df_gold["data_publicacao"]).dt.days

# Coorte = mês de publicação; madura = já teve 90 dias para ser fiscalizada
df_gold["coorte"] = df_gold["data_publicacao"].dt.to_period("M").dt.to_timestamp()
df_gold["coorte_madura"] = (df_gold["data_publicacao"] + pd.Timedelta(days=90)) <= reference_date

# Marcadores do funil
discarded = ["falso positivo", "tardio (a definir)", "outro órgão"]
df_gold["descartado"] = df_gold["status"].isin(discarded)
df_gold["fiscalizado"] = df_gold["status"].str.startswith("fiscalizado")
df_gold["fiscalizado_90d"] = df_gold["dias_ate_fiscalizacao"].le(90)
df_gold["com_infracao"] = df_gold["status"].eq("fiscalizado com infração")

# Fiscalização antes da publicação: sai das métricas de tempo e fica marcada
df_gold["data_inconsistente"] = df_gold["dias_ate_fiscalizacao"] < 0
df_gold.loc[df_gold["data_inconsistente"], "dias_ate_fiscalizacao"] = np.nan

In [ ]:
# Regionalização do ES (Lei 9.768/2011, atualizada pela Lei 11.174/2020)
reference_dir = Path("../referencias")
df_regions = pd.read_csv(reference_dir / "regionalizacao_es.csv")

# Chave de junção: nome sem acentos, em minúsculas e sem espaços nas pontas
def name_key(series):
    return (series.str.normalize("NFKD").str.encode("ascii", "ignore")
                  .str.decode("ascii").str.lower().str.strip())

df_regions["chave_municipio"] = name_key(df_regions["nome_municipio"])
df_gold["chave_municipio"] = name_key(df_gold["municipio"])

# Municípios dos alertas sem correspondência na regionalização
unmatched = df_gold.loc[~df_gold["chave_municipio"].isin(df_regions["chave_municipio"]), "municipio"]
print(unmatched.nunique(), "nomes sem correspondência:", sorted(unmatched.unique()))

In [ ]:
# Leva o código do IBGE para os alertas (muitos alertas para um município)
df_gold = (df_gold
           .merge(df_regions[["chave_municipio", "cd_municipio"]],
                  on="chave_municipio", how="left", validate="many_to_one")
           .drop(columns="chave_municipio"))
df_gold["cd_municipio"] = df_gold["cd_municipio"].astype("Int64")

# Só os alertas do ES precisam ter município na regionalização
in_es = df_gold["uf"].eq("ES")
assert df_gold.loc[in_es, "cd_municipio"].notna().all(), "Há alerta do ES sem município na regionalização"

# Reconciliação com a Nota Técnica: municípios do ES com alerta que gerou infração
n_municipalities = df_gold.loc[df_gold["com_infracao"] & in_es, "cd_municipio"].nunique()
print(n_municipalities, "municípios do ES com alerta que gerou infração (NT: 69)")

# Tabela de municípios para o Power BI
gold_dir.mkdir(parents=True, exist_ok=True)
df_regions.drop(columns="chave_municipio").to_parquet(gold_dir / "municipios.parquet", index=False)

In [ ]:
# Checagens informativas
negative = df_gold["dias_ate_fiscalizacao"] < 0
print(negative.sum(), "alertas fiscalizados ANTES da publicação")
print(df_gold["coorte_madura"].sum(), "alertas em coortes maduras")
print("Mediana de dias até a fiscalização:", df_gold["dias_ate_fiscalizacao"].median())

# Salva na camada ouro
gold_dir.mkdir(parents=True, exist_ok=True)
df_gold.to_parquet(gold_dir / "alertas.parquet", index=False)
df_gold.info()

In [ ]:
# Tabelas de fiscalização para o modelo do Power BI
df_infractions = pd.read_parquet(silver_dir / "infracoes.parquet")
df_bridge = pd.read_parquet(silver_dir / "ponte_alerta_fiscalizacao.parquet")

# Fiscalizações: um número por linha, com a data
df_inspections = (pd.concat([df_bridge[["id_fiscalizacao"]], df_infractions[["id_fiscalizacao"]]])
                  .dropna().drop_duplicates().reset_index(drop=True))
inspection_dates = (df_responses.dropna(subset=["id_fiscalizacao"])
                    .groupby("id_fiscalizacao", as_index=False)["data_fiscalizacao"].min())
df_inspections = df_inspections.merge(inspection_dates, on="id_fiscalizacao",
                                      how="left", validate="one_to_one")

# Checagens
assert df_bridge["id_alerta"].isin(df_gold["id_alerta"]).all(), "Ponte aponta para alerta inexistente"
print(df_infractions["id_fiscalizacao"].isna().sum(), "infrações sem número de fiscalização")
print(len(df_inspections), "fiscalizações |", df_inspections["data_fiscalizacao"].isna().sum(), "sem data")

# Salva as três tabelas na camada ouro
df_inspections.to_parquet(gold_dir / "fiscalizacoes.parquet", index=False)
df_infractions.to_parquet(gold_dir / "infracoes.parquet", index=False)
df_bridge.to_parquet(gold_dir / "ponte_alerta_fiscalizacao.parquet", index=False)